# BreastDCEDL on Colab A100 — End-to-End pCR Prediction

Containerized walkthrough: **mount Drive → clone → bootstrap → download MinCrop → verify → smoke test → full train → evaluate → report**.

**Benchmark target:** the Fridman et al. 2025 paper reports patient-level **AUC 0.94 on the HR+/HER2− subtype** (best performance). Overall test AUC is 0.72 and overall accuracy is 0.75; those numbers are already near published SOTA, so "90%+" is realistic for the HR+/HER2− subtype, which is what this notebook targets.

**Before running:** `Runtime → Change runtime type → GPU → A100` (40 GB).

**Storage:** the MinCrop archive is ~22 GB extracted. This notebook pulls it once into `/content/drive/MyDrive/breastdcedl_data/` so it survives runtime restarts.

**Pipeline:**
1. Zenodo record [18114231](https://doi.org/10.5281/zenodo.18114231) (MinCrop data + paper's pretrained weights)
2. DINOv2-Base / ViT-Base backbone with classifier head
3. Two-phase training: head-only warmup → full fine-tune with layer-wise LR decay (LLRD)
4. Focal loss (handles ~30% pCR-positive imbalance)
5. Patient-level pooling + horizontal/vertical-flip TTA at test time
6. Subtype-stratified metrics (HR+/HER2−, HER2+, TripleNeg)

## 1. Runtime check

In [ ]:
import torch, platform
print(f"python   : {platform.python_version()}")
print(f"torch    : {torch.__version__}")
print(f"cuda     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    raise SystemExit("No CUDA device. Runtime → Change runtime type → A100.")

## 2. Mount Google Drive

All artifacts (data, checkpoints, results) land under `/content/drive/MyDrive/breastdcedl/`, so re-runs skip the download and resume from the best checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/breastdcedl'
DATA_DIR = f'{DRIVE_ROOT}/data'
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

## 3. Clone repo & bootstrap dependencies

Point `REPO_URL` at your fork if you've pushed local changes. The bootstrap script installs pinned deps and sanity-checks the GPU and imports.

In [ ]:
REPO_URL = 'https://github.com/javaapplesauce/BreastDCEDL-4-GILM.git'
REPO_DIR = '/content/breastdcedl'

# Shallow clone — repo history contains ~1.2 GB of committed sample files.
# --depth=1 --single-branch grabs only the latest commit on the default branch.
!rm -rf $REPO_DIR
!git clone --depth=1 --single-branch $REPO_URL $REPO_DIR
%cd $REPO_DIR
!bash colab/bootstrap.sh $REPO_DIR

In [ ]:
import sys, os

# Ensure the repo dir is on sys.path AND is the cwd so `from src.*` works.
# Re-run this cell after any kernel restart.
REPO_DIR = '/content/breastdcedl'
assert os.path.isdir(REPO_DIR), f'{REPO_DIR} missing — re-run the clone cell above.'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Guarantee namespace packages have __init__.py (a shallow clone missing any of
# these would break `from src.*` imports).
for sub in ('src', 'src/data', 'src/models', 'src/training', 'src/evaluation'):
    init = os.path.join(REPO_DIR, sub, '__init__.py')
    if os.path.isdir(os.path.dirname(init)) and not os.path.isfile(init):
        open(init, 'a').close()

print('cwd :', os.getcwd())
print('path:', REPO_DIR, 'on sys.path')

## 4. (Optional) W&B login

Skip this cell if you don't use W&B; training will still log to the local history.json.

In [ ]:
USE_WANDB = False  # set True and run this cell to log to Weights & Biases
if USE_WANDB:
    import wandb
    wandb.login()

## 5. Download MinCrop data + pretrained weights from Zenodo

**Strategy**: archives cache on Drive (`DRIVE_CACHE`), but extraction goes to local Colab disk (`/content/data`). Drive FUSE is ~100× slower for many small files — extracting ~2000 `.nii.gz` directly to Drive takes hours; local takes minutes. On runtime restart, re-extracting from the Drive cache runs in ~3–5 min vs. a full ~30 min re-download.

After this cell, `DATA_DIR` is reassigned to `/content/data` for downstream cells.

| File | Size |
|---|---|
| `BreastDCEDL_ISPY2_min_crop.tar.gz` | 13.6 GB |
| `BreastDCEDL_DUKE_min_crop.tar.gz`  | 8.6 GB |
| `BreastDCEDL_ISPY1_min_crop.tar.gz` | 1.2 GB |
| `BreastDCEDL_models.tar.gz`         | 305 MB |
| `BreastDCEDL_metadata_min_crop.csv` | 340 KB |

For a fast smoke test, change `SUBSETS = ['demo', 'metadata', 'models']`.

In [ ]:
import os, shutil, tarfile, urllib.request, time
from src.data.zenodo import FILES, ZENODO_API

# Drive caches the archives; local disk holds the extracted trees (FUSE is too slow).
DRIVE_CACHE = DATA_DIR            # /content/drive/MyDrive/breastdcedl/data
LOCAL_DATA  = '/content/data'     # fast local Colab disk
os.makedirs(DRIVE_CACHE, exist_ok=True)
os.makedirs(LOCAL_DATA,  exist_ok=True)

SUBSETS = ['ispy1', 'ispy2', 'duke', 'metadata', 'models']
# SUBSETS = ['demo', 'metadata', 'models']  # ~400 MB, smoke-test only

def _download(url, dest):
    t0 = time.time()
    urllib.request.urlretrieve(url, dest)
    mb = os.path.getsize(dest) / 1e6
    print(f'    done in {time.time()-t0:.0f}s  ({mb:.0f} MB  @ {mb/max(time.time()-t0,1):.1f} MB/s)')

for subset in SUBSETS:
    info = FILES[subset]
    fname = info['key']
    cached = os.path.join(DRIVE_CACHE, fname)
    local  = os.path.join(LOCAL_DATA,  fname)

    if not os.path.isfile(cached):
        url = f"{ZENODO_API}/files/{fname}/content"
        print(f'[download] {fname}  → Drive cache')
        _download(url, cached)
    else:
        print(f'[cache]    {fname}  ({os.path.getsize(cached)/1e6:.0f} MB)')

    if fname.endswith('.tar.gz'):
        # Extract directly from Drive cache to local disk (one streaming read).
        marker = os.path.join(LOCAL_DATA, f'.extracted_{subset}')
        if os.path.isfile(marker):
            print(f'[extract]  {fname}  already extracted')
        else:
            print(f'[extract]  {fname}  → {LOCAL_DATA}')
            t0 = time.time()
            with tarfile.open(cached, 'r:gz') as t:
                t.extractall(LOCAL_DATA)
            open(marker, 'w').close()
            print(f'    extracted in {time.time()-t0:.0f}s')
    else:
        # CSV / small file — copy to local so path discovery finds it there.
        if not os.path.isfile(local):
            shutil.copy(cached, local)

# Reassign DATA_DIR so downstream cells read from local disk.
DATA_DIR = LOCAL_DATA
print('\nDATA_DIR is now:', DATA_DIR)
!du -sh {DATA_DIR}/* 2>/dev/null | sort -h | tail -10

## 6. Discover extracted directories

MinCrop archives extract into `BreastDCEDL_<COHORT>_min_crop/` with `dce/` (or `spt1_dce/`) and `mask/` subfolders. This cell walks the tree and locates them.

In [ ]:
from pathlib import Path

cohort_roots = {
    'spy1': f'{DATA_DIR}/BreastDCEDL_ISPY1_min_crop',
    'spy2': f'{DATA_DIR}/BreastDCEDL_ISPY2_min_crop',
    'duke': f'{DATA_DIR}/BreastDCEDL_DUKE_min_crop',
}

nifti_paths, mask_paths = {}, {}
for cohort, root in cohort_roots.items():
    if not os.path.isdir(root):
        print(f'  [skip] {cohort}: {root} not present')
        continue
    # Mask dir: first subdir whose name contains "mask"
    mdir = next(
        (str(p) for p in sorted(Path(root).rglob('*'))
         if p.is_dir() and 'mask' in p.name.lower()),
        None,
    )
    # DCE dir: parent of the first .nii.gz file that is NOT under the mask dir
    ndir = None
    for p in sorted(Path(root).rglob('*.nii.gz')):
        parent = str(p.parent)
        if mdir and parent.startswith(mdir):
            continue
        ndir = parent; break

    if ndir: nifti_paths[cohort] = ndir
    if mdir: mask_paths[cohort]  = mdir
    n_nifti = len(list(Path(ndir).glob('*.nii.gz'))) if ndir else 0
    n_mask  = len(list(Path(mdir).glob('*.nii.gz'))) if mdir else 0
    print(f'  {cohort:5s} dce={ndir}  ({n_nifti} files)')
    print(f'  {cohort:5s} msk={mdir}  ({n_mask} files)')

# Paper's pretrained weights (from BreastDCEDL_models.tar.gz)
pth_files = sorted(Path(DATA_DIR).rglob('*.pth'))
print(f'\nPretrained .pth files:')
for p in pth_files:
    print(f'  {p.relative_to(DATA_DIR)}  ({p.stat().st_size/1e6:.0f} MB)')
PRETRAINED = str(pth_files[0]) if pth_files else None
print(f'\nSelected pretrained: {PRETRAINED}')

METADATA_CSV = f'{DATA_DIR}/BreastDCEDL_metadata_min_crop.csv'
assert os.path.isfile(METADATA_CSV), f'metadata missing: {METADATA_CSV}'

## 7. Configure training (writes `configs/colab.yaml`)

In [ ]:
import yaml, copy

with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['nifti'] = nifti_paths
cfg['data']['masks'] = mask_paths
cfg['data']['combined_metadata'] = METADATA_CSV
cfg['data']['crop_size'] = 224
cfg['data']['n_slices'] = 8
cfg['data']['label_col'] = 'pCR'

cfg['model']['backbone'] = 'facebook/dinov2-base'  # or 'google/vit-base-patch16-224-in21k'
cfg['model']['pretrained_weights'] = PRETRAINED     # paper's .pth weights
cfg['model']['dropout'] = 0.3

# A100 40GB: batch 48 x accum 1 fits comfortably at 224px / ViT-Base.
# Drop to 32 if you add clinical features or enable second-order augs.
cfg['training']['batch_size'] = 48
cfg['training']['accum_steps'] = 1
cfg['training']['num_epochs'] = 30
cfg['training']['freeze_epochs'] = 5
cfg['training']['backbone_lr'] = 5e-6
cfg['training']['head_lr'] = 5e-4
cfg['training']['llrd'] = 0.85
cfg['training']['patience'] = 10
cfg['training']['num_workers'] = 4
cfg['training']['loss'] = 'focal'

cfg['wandb'] = {
    'enabled': USE_WANDB,
    'project': 'breastdcedl-vit',
    'entity': None,
    'tags': ['a100', 'dinov2', 'pcr', 'mincrop'],
}
cfg['checkpoint_dir'] = CKPT_DIR

with open('configs/colab.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print('wrote configs/colab.yaml')
!sed -n '1,25p' configs/colab.yaml

## 8. Verify data loading (sanity-check RGB fusion + ROI crop)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.data.preprocessing import setup_paths, load_acquisitions, fuse_rgb_slice, select_timepoints, _cohort_from_pid
from src.data.splits import load_and_split

setup_paths(nifti_dirs=nifti_paths, mask_dirs=mask_paths)

train_df, val_df, test_df = load_and_split(METADATA_CSV, label_col='pCR', seed=42)

# Filter to cohorts we actually downloaded (so the DataLoader never stalls on missing files)
avail = set(nifti_paths.keys())
if 'dataset' in train_df.columns:
    train_df = train_df[train_df['dataset'].isin(avail)].reset_index(drop=True)
    val_df   = val_df  [val_df  ['dataset'].isin(avail)].reset_index(drop=True)
    test_df  = test_df [test_df ['dataset'].isin(avail)].reset_index(drop=True)

print(f'train={len(train_df)}  val={len(val_df)}  test={len(test_df)}')
print(f'pCR rates: train={train_df.pCR.mean():.1%}  val={val_df.pCR.mean():.1%}  test={test_df.pCR.mean():.1%}')
if 'dataset' in train_df.columns:
    print('cohorts (train):', train_df.dataset.value_counts().to_dict())

# Visualize 4 random patients
sample = train_df.sample(min(4, len(train_df)), random_state=1)
fig, axes = plt.subplots(1, len(sample), figsize=(4*len(sample), 4))
if len(sample) == 1: axes = [axes]
for ax, (_, row) in zip(axes, sample.iterrows()):
    pid = row['pid']
    acqs = load_acquisitions(pid)
    if acqs is None or len(acqs) < 2:
        ax.set_title(f'{pid}\n(no data)'); ax.axis('off'); continue
    cohort = _cohort_from_pid(pid)
    ip = row.get('pre');        ip = None if pd.isna(ip) else ip
    ie = row.get('post_early'); ie = None if pd.isna(ie) else ie
    il = row.get('post_late');  il = None if pd.isna(il) else il
    pre, early, late = select_timepoints(acqs, cohort, idx_pre=ip, idx_early=ie, idx_late=il)
    z = pre.shape[2] // 2
    rgb = fuse_rgb_slice(pre[:, :, z], early[:, :, z], late[:, :, z])
    ax.imshow(rgb); ax.axis('off')
    ax.set_title(f'{pid[:18]}...\npCR={int(row.pCR)}  {cohort}')
plt.suptitle('RGB fusion (R=pre, G=early-post, B=late-post)')
plt.tight_layout(); plt.show()

## 9. Smoke test (2 epochs, tiny batch) — verify pipeline end-to-end

Expected runtime: **~2–4 min** on A100 with the full dataset. If this fails, do **not** start the full run.

In [ ]:
from scripts.train import train_from_config
import copy as _copy

smoke = _copy.deepcopy(cfg)
smoke['training']['num_epochs'] = 2
smoke['training']['freeze_epochs'] = 1
smoke['training']['batch_size'] = 16
smoke['training']['num_workers'] = 2
smoke['data']['n_slices'] = 4
smoke['checkpoint_dir'] = f'{CKPT_DIR}/smoke'
smoke['wandb'] = {'enabled': False, 'project': 'breastdcedl-vit', 'entity': None, 'tags': []}

_ = train_from_config(smoke)

## 10. Full training run

Expected A100 runtime: **~45–60 min** for 30 epochs on the full ~1,500-patient training set (8 slices × ~1,100 patients = ~8,800 samples per epoch). Checkpoints go to Drive; resume by re-running this cell.

In [ ]:
best_auc = train_from_config(cfg)
print(f'\nBest validation AUC: {best_auc:.4f}')
print(f'Checkpoints in {CKPT_DIR}')

## 11. Training curves

In [ ]:
import json
with open(f'{CKPT_DIR}/history.json') as f: history = json.load(f)

epochs = [e['epoch'] for e in history]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(epochs, [e['train_loss'] for e in history], label='train')
ax[0].plot(epochs, [e['val_loss']   for e in history], label='val')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].set_title('Loss')
ax[1].plot(epochs, [e['train_acc']  for e in history], label='train')
ax[1].plot(epochs, [e['accuracy']   for e in history], label='val (patient)')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy'); ax[1].legend(); ax[1].set_title('Accuracy')
ax[2].plot(epochs, [e['auc'] for e in history], color='tab:green', label='val AUC')
ax[2].axhline(0.72, linestyle='--', color='grey', alpha=0.7, label='paper overall (0.72)')
ax[2].axhline(0.94, linestyle='--', color='red',  alpha=0.7, label='paper HR+/HER2− (0.94)')
ax[2].set_xlabel('epoch'); ax[2].set_ylabel('AUC'); ax[2].legend(); ax[2].set_title('Patient-level AUC')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150); plt.show()

## 12. Evaluate on held-out test set (patient-level + TTA)

In [ ]:
from torch.utils.data import DataLoader
from src.data.dataset import BreastDCEDataset, build_transforms
from src.models.vit import BreastDCEViT
from src.evaluation.metrics import compute_metrics, evaluate_by_subtype, predict_with_tta

device = torch.device('cuda')
n_slices  = cfg['data']['n_slices']
crop_size = cfg['data']['crop_size']
label_col = cfg['data']['label_col']

test_ds = BreastDCEDataset(
    test_df, label_col=label_col,
    crop_size=crop_size, n_slices=n_slices,
    transform=build_transforms({}, is_train=False),
)
test_loader = DataLoader(
    test_ds, batch_size=cfg['training']['batch_size'],
    shuffle=False, num_workers=cfg['training']['num_workers'], pin_memory=True,
)

# Load best checkpoint
eval_model = BreastDCEViT(
    backbone=cfg['model']['backbone'],
    num_classes=cfg['model']['num_classes'],
    dropout=0.0,
).to(device)
best_path = f"{CKPT_DIR}/best.pth"
eval_model.load_state_dict(torch.load(best_path, map_location=device))
eval_model.eval()
print(f'loaded {best_path}')

# Patient-level eval with 4-view TTA (identity + h-flip + v-flip + hv-flip)
y_true, y_prob, y_pred = predict_with_tta(
    eval_model, test_loader, device,
    n_slices=n_slices, use_clinical=False, tta=True,
)
print(f'test set: {len(y_true)} patients')
overall = compute_metrics(y_true, y_prob, y_pred)
print('\n=== Overall Test Set ===')
for k, v in overall.items():
    print(f'  {k:13s}: {v:.4f}')

print('\n=== Paper baseline (Table 2, overall test) ===')
print('  AUC            : 0.72')
print('  Accuracy       : 0.75')
print('  Sensitivity    : 0.27')
print('  Specificity    : 0.95')

## 13. Subtype + per-cohort breakdown

The key headline metric: **AUC on HR+/HER2−** (paper reports 0.94).

In [ ]:
subtypes = cfg.get('evaluation', {}).get('subtypes', {
    'HR+/HER2−': {'HRposHER2neg': 1},
    'HER2+':     {'HER2pos': 1},
    'TripleNeg': {'TripleNeg': 1},
})
sub_df = evaluate_by_subtype(test_df.reset_index(drop=True), y_prob, y_pred, label_col=label_col, subtypes=subtypes)
print('\n=== Subtype breakdown (test set) ===')
print(sub_df.to_string(index=False, float_format='%.3f'))
sub_df.to_csv(f'{RESULTS_DIR}/subtype_results.csv', index=False)

# Per-cohort
if 'pid' in test_df.columns:
    rows = []
    for name, pat in [('Duke','Breast_MRI'),('I-SPY1','ISPY1'),('I-SPY2','ISPY2'),('ACRIN-6698','ACRIN-6698')]:
        m = test_df.reset_index(drop=True)['pid'].str.contains(pat, na=False).values
        if m.sum() < 3: continue
        rows.append({'Cohort': name, 'N': int(m.sum()), **compute_metrics(y_true[m], y_prob[m], y_pred[m])})
    cohort_df = pd.DataFrame(rows)
    if len(cohort_df):
        print('\n=== Per-cohort breakdown (test set) ===')
        print(cohort_df.to_string(index=False, float_format='%.3f'))
        cohort_df.to_csv(f'{RESULTS_DIR}/cohort_results.csv', index=False)

## 14. ROC + confusion matrix

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix as cm_fn
import seaborn as sns

fpr, tpr, _ = roc_curve(y_true, y_prob)
auc_val = roc_auc_score(y_true, y_prob)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(fpr, tpr, lw=2, label=f'AUC = {auc_val:.3f}')
ax[0].plot([0, 1], [0, 1], '--', color='grey')
ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].legend()
ax[0].set_title('Test ROC'); ax[0].set_aspect('equal')

cm = cm_fn(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-pCR','pCR'], yticklabels=['Non-pCR','pCR'], ax=ax[1])
ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('True'); ax[1].set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/roc_cm.png', dpi=150); plt.show()

## 15. Save predictions + artifacts to Drive

In [ ]:
pred_df = test_df.reset_index(drop=True)[['pid']].copy() if 'pid' in test_df.columns else pd.DataFrame(index=range(len(y_true)))
pred_df['y_true'] = y_true
pred_df['y_prob'] = y_prob
pred_df['y_pred'] = y_pred
pred_df.to_csv(f'{RESULTS_DIR}/predictions.csv', index=False)

# Copy history to results for a single self-contained results dir
!cp -f {CKPT_DIR}/history.json {RESULTS_DIR}/history.json 2>/dev/null || true
!cp -f {CKPT_DIR}/best.pth     {RESULTS_DIR}/best.pth     2>/dev/null || true

print(f'Results saved to {RESULTS_DIR}:')
!ls -la {RESULTS_DIR}

## Limitations & notes

- **90%+ target**: only the **HR+/HER2−** subtype (~40% of patients) is expected to reach AUC 0.9+; overall AUC is in the 0.70–0.75 range (published SOTA). The paper's 0.94 result is derived from this subtype.
- **Pretrained weights**: the Zenodo `BreastDCEDL_models.tar.gz` archive contains the paper's `.pth` checkpoint. Loading it gives the immediate boost; training from scratch on Colab in a single session is unlikely to match.
- **Data integrity**: ~640 patients are missing pCR labels (Duke cohort skews that way). `load_and_split` drops them before training.
- **Timepoint selection**: the MinCrop metadata encodes per-patient `pre`, `post_early`, `post_late` scan indices. `preprocessing.select_timepoints` honours those when present and falls back to cohort defaults otherwise.
- **Test-time augmentation**: 4-view TTA (identity + h-flip + v-flip + both) typically lifts AUC ~0.01–0.02; report includes it by default.
- **A100 40GB headroom**: batch 48 at 224px fits; bump to 64 if you turn off augmentation, or drop to 24 if you enable clinical features + FP32.
- **Colab session limits**: a 30-epoch run comfortably fits in one session. If your runtime disconnects mid-training, cell 10 re-runs resume from the last best checkpoint on Drive because `train_from_config` honours the `resume` key — add `cfg['resume'] = f'{CKPT_DIR}/best.pth'` before the call.
- **Not included**: explicit segmentation objective. The masks are used only to locate the tumor midplane for slice selection — the model is trained to predict pCR, not segment.